## **LLM Based Conditional Workflow**

In [1]:
from langgraph.graph import StateGraph, START,END
from langchain_openai import ChatOpenAI
from typing import TypedDict,Literal
from dotenv import load_dotenv
from pydantic import BaseModel, Field

In [2]:
load_dotenv()

True

In [3]:
model = ChatOpenAI(model='gpt-4o-mini')

In [5]:
# Test the model 
model.invoke("WHAT IS GENAI ?")

AIMessage(content="GenAI, or Generative AI, refers to a subset of artificial intelligence that focuses on creating new content, data, or solutions by learning from existing examples. This can include generating text, images, music, video, and other forms of media. \n\nGenerative AI models use techniques such as deep learning, neural networks, and natural language processing to produce content that can be indistinguishable from that created by humans. Some well-known examples include:\n\n1. **Text Generation**: Models like OpenAI's GPT (Generative Pre-trained Transformer) can write essays, answer questions, or engage in conversations.\n\n2. **Image Generation**: Tools like DALL-E and Midjourney can create images based on textual descriptions.\n\n3. **Music Composition**: AI can generate original pieces of music by learning from a variety of musical styles.\n\n4. **Video Creation**: AI can even create synthetic media, including animations and deepfake videos.\n\nGenerative AI has a wide 

In [7]:
class SentimentSchema(BaseModel):
    sentiment:Literal["Positive","negative"] = Field(description="Sentiment of the review")

In [ ]:
class DiagnosisSchema(BaseModel):
    
    issue_type: Literal["UX","Performance","Bug","Support","Other"] = Field(description="Thecategory of the issue in review")
    tone: Literal["angry","frustrated","disappointed","calm"] = Field(description="The emotional tone express by the user")
    urgency: Literal["Low","Medium","High"] = Field(description="Hw urgent urgent the issue appears to be ")

In [17]:
structured_model1 = model.with_structured_output(SentimentSchema)
structured_model2 = model.with_structured_output(DiagnosisSchema)

# testing the model 
prompt = "What is the sentiment of the review gibven by the user-The software is really bad"
structured_model1.invoke(prompt)

SentimentSchema(sentiment='negative')

In [39]:
# Build the state of the class 
class Reviews_state(TypedDict):

    review:str 
    sentiment:Literal["Positive","Negative"]
    diagnosis:dict 
    response:str 

In [40]:
# defining grpah 
graph = StateGraph(Reviews_state)

In [41]:
# building nodes of the graph 

def find_sentiment(state:Reviews_state) -> Reviews_state:

    prompt = f"For the given review by the user find out the sentiment \n{state['review']}"
    sentiment = structured_model1.invoke(prompt).sentiment

    return {"sentiment":sentiment}

def check_sentiment(state:Reviews_state) -> Literal['Positive_response',"run_diagnosis"]:

    if state['sentiment']=="Positive":
        return 'Positive_response'
    else:
        return 'run_diagnosis'
    

def positive_response(state:Reviews_state) -> Reviews_state:

    prompt  = f"""Kindly write a warm thank you to the user for such nice review \n {state['review']}
     and also ask user to leave feedback on our website"""
    
    response =model.invoke(prompt).content

    return {f"response":response}


def run_diagnosis(state:Reviews_state):

    prompt = f"""Diagnose this negative review \n{state['review']}\n
                and return issue_type, tone, and urgency"""
    
    response = model.invoke(prompt)

    return {"response":response}

def negative_response(state:Reviews_state):

    diagnosis = state['review']

    prompt = f"""You are support assistant. The user had a {diagnosis['issue_type']} issue, sounded {diagnosis['tone']}
                     and marked urgency as {diagnosis['urgency']}. 
                     write an empathetic, helpful resolution message to the user.
                     """
    
    response = model.invoke(prompt).content

    return {"response":response}

In [42]:
# adding nodes to the graph 
graph.add_node("find_sentiment",find_sentiment)
graph.add_node("positive_response",positive_response)
graph.add_node("run_diagnosis",run_diagnosis)
graph.add_node("negative_response",negative_response)



In [43]:
# adding the edges of the gaph 
graph.add_edge(START,"find_sentiment")
graph.add_edge("find_sentiment","find_sentiment")
graph.add_edge("positive_response",END)
graph.add_edge("run_diagnosis","negative_response")
graph.add_edge('negative_response',END)
workflow = graph.compile()

In [53]:
workflow

ValueError: Failed to reach https://mermaid.ink API while trying to render your graph. Status code: 400.

To resolve this issue:
1. Check your internet connection and try again
2. Try with higher retry settings: `draw_mermaid_png(..., max_retries=5, retry_delay=2.0)`
3. Use the Pyppeteer rendering method which will render your graph locally in a browser: `draw_mermaid_png(..., draw_method=MermaidDrawMethod.PYPPETEER)`